# Module 4 - SHAP analysis notebook
Bank of Kigali credit-risk model (LendingClub stand-in data). Run this **after** `python -m src.module4.run_all`.

It loads the saved model and the 2015 test set, then shows: global importance, local explanations, counterfactual what-ifs and sensitivity tests. Use **Run All**, then save, so the outputs are stored in the notebook.

*SHAP values are model associations, not proof of cause. For tree models they are in the model's output units (log-odds for LightGBM, probability for random forest); contributions are summed per original feature (one-hot columns are added back together).*

In [1]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import joblib, numpy as np, pandas as pd
import matplotlib.pyplot as plt, shap
from src.module4 import config, data, scenarios, plots
from src.module4.explain import Explainer, global_importance
from src.module4.model import feature_frame

DEMO = os.environ.get("M4_DEMO") == "1"     # synthetic dry run only; leave unset for your real results
train, val, test = data.prepare(demo=DEMO)
model = joblib.load(config.MODEL_FILE)
Xte = feature_frame(test).reset_index(drop=True)
yte = test[config.TARGET].reset_index(drop=True)
print("model:", model.name, "| test loans:", len(Xte), "| default rate:", round(yte.mean(), 3))
if DEMO: print("*** DEMO DATA - do not quote ***")

/usr/local/lib/python3.13/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


model: lightgbm | test loans: 30835 | default rate: 0.255


## 1. Global feature importance (SHAP)

In [2]:
expl = Explainer(model)
idx = np.random.default_rng(config.SEED).choice(len(Xte), min(2000, len(Xte)), replace=False)
Xs = Xte.iloc[idx].reset_index(drop=True)
agg, base = expl.values(Xs)
imp = global_importance(agg)
imp.head(10)

/usr/local/lib/python3.13/dist-packages/shap/explainers/_tree.py:578: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


,feature,mean_abs_shap
0,int_rate,0.451427
1,term,0.185954
2,annual_inc,0.174618
3,dti,0.107860
4,loan_to_income,0.094033
5,revol_util,0.062532
6,open_acc,0.056506
7,total_acc,0.046781
8,inq_last_6mths,0.038810
9,installment_to_income,0.033692


In [3]:
enc = Xs.copy()
for c in config.CATEGORICAL: enc[c] = enc[c].astype("category").cat.codes
shap.plots.beeswarm(shap.Explanation(values=agg, data=enc.astype(float).to_numpy(), feature_names=list(enc.columns)), max_display=15)

**How to read it:** each dot is a loan; right of zero raises predicted default risk, left lowers it; red = high feature value, blue = low.

In [4]:
top = imp.head(5)
for _, r in top.iterrows():
    col = config.FEATURES.index(r.feature)
    if r.feature in config.NUMERIC:
        corr = np.corrcoef(Xs[r.feature].astype(float).fillna(Xs[r.feature].median()), agg[:, col])[0, 1]
        direction = "higher values raise risk" if corr > 0 else "higher values lower risk"
    else:
        direction = "depends on the category"
    print(f"{r.feature:<24} mean |SHAP| = {r.mean_abs_shap:.3f}  ({direction})")

int_rate                 mean |SHAP| = 0.451  (higher values raise risk)
term                     mean |SHAP| = 0.186  (higher values raise risk)
annual_inc               mean |SHAP| = 0.175  (higher values lower risk)
dti                      mean |SHAP| = 0.108  (higher values raise risk)
loan_to_income           mean |SHAP| = 0.094  (higher values raise risk)


## 2. Local explanations - three applicants

In [5]:
raw = model.raw(Xs)
picks = {"highest risk": int(np.argmax(raw)), "medium risk": int(np.argsort(raw)[len(raw)//2]), "lowest risk": int(np.argmin(raw))}
for label, i in picks.items():
    print(f"\n{label}: default probability {model.predict_proba(Xs.iloc[[i]])[0]:.3f}, tier {model.tier(raw[i:i+1])[0]}")
    for r in expl.reasons(Xs.iloc[[i]]): print("   ", r["feature"], "=", r["value"], "->", r["effect"], f"({r['contribution']:+.3f})")
    shap.plots.waterfall(shap.Explanation(values=agg[i], base_values=base, data=Xs.iloc[i].astype(object).to_numpy(), feature_names=list(Xs.columns)), max_display=10)


highest risk: default probability 0.733, tier high
    int_rate = 25.99 -> raises risk (+0.883)
    term = 60 -> raises risk (+0.332)
    annual_inc = 25000.0 -> raises risk (+0.251)

medium risk: default probability 0.235, tier low
    annual_inc = 114000.0 -> lowers risk (-0.248)
    dti = 28.26 -> raises risk (+0.194)
    inq_last_6mths = 3.0 -> raises risk (+0.139)


/usr/local/lib/python3.13/dist-packages/shap/explainers/_tree.py:578: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/shap/explainers/_tree.py:578: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(



lowest risk: default probability 0.000, tier low
    int_rate = 5.32 -> lowers risk (-1.266)
    revol_util = 4.4 -> lowers risk (-0.373)
    annual_inc = 250000.0 -> lowers risk (-0.349)


/usr/local/lib/python3.13/dist-packages/shap/explainers/_tree.py:578: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


## 3. Counterfactual what-ifs
What would need to change for a different outcome? Flagged 60-month applicants, with the instalment recomputed at the same interest rate. These are examples, not offers: a shorter term raises the payment, so affordability must be checked.

In [6]:
cf = scenarios.counterfactuals(model, test.reset_index(drop=True))
cf.round(3)

,applicant,scenario,term,loan_amnt,installment,default_probability,tier
0,9372,as applied,60,21700.0,565.190,0.442,high
1,9372,term 60 -> 36 months,36,21700.0,797.522,0.325,medium
2,9372,amount -10%,60,19530.0,508.663,0.427,high
3,9372,amount -20%,60,17360.0,452.145,0.400,high
4,11875,as applied,60,20800.0,537.740,0.490,high
5,11875,term 60 -> 36 months,36,20800.0,760.764,0.383,high
6,11875,amount -10%,60,18720.0,483.961,0.461,high
7,11875,amount -20%,60,16640.0,430.187,0.442,high
8,7412,as applied,60,35000.0,911.590,0.513,high
9,7412,term 60 -> 36 months,36,35000.0,1286.325,0.383,medium


## 4. Sensitivity analysis
Average change in predicted default probability (percentage points) when one input moves by 10%.

In [7]:
sens = scenarios.sensitivity(model, test.reset_index(drop=True))
print("AUC clean:", round(sens.attrs["auc_clean"], 4), "| AUC with 5% random noise:", round(sens.attrs["auc_noise5"], 4))
sens.round(3)

AUC clean: 0.6872 | AUC with 5% random noise: 0.6852


,input,scenario,change_pp
0,annual_inc,income -10%,1.188
1,annual_inc,income +10%,-1.116
2,int_rate,interest rate +10%,2.201
3,dti,debt-to-income +10%,0.365
4,loan_amnt,loan amount +10%,0.523
5,revol_util,credit use +10%,0.195


## 5. Business reading
1. The strongest drivers are listed in section 1; interest rate and grade are set by the lender, so the model partly repeats an earlier lending judgement.
2. Each API response returns the three largest contributions for the applicant, in the same terms as section 2.
3. These explanations support a human reviewer; they do not replace one.